# 04 Maps

Phase 4: kharita draws the map itself, with the legend in the notebook and in exported HTML.

In [1]:
import warnings
import webbrowser
from pathlib import Path

import geopandas as gpd
import pandas as pd

import kharita as kh

warnings.filterwarnings("ignore", module="lonboard")
states = gpd.read_file("../data/processed/sudan_states.gpkg")
SOURCE = "U.S. Census Bureau via HDX, CC BY 4.0. 2008 census, retabulated."

In [2]:
# One line: classified choropleth, outlines, hover, tooltip and legend
kh.map(
    states,
    "preset-choropleth fill-[density] ramp-ylorrd tooltip-[key,pop,density]",
    legend={"title": "People per km2", "note": SOURCE},
)

In [3]:
# Your own breaks and labels
kh.map(
    states,
    "preset-choropleth fill-[pop] ramp-blues scheme-[1000000,2000000,3000000] legend-tl",
    legend={"title": "Population", "labels": ["Under 1 million", "1 to 2 million", "2 to 3 million", "Over 3 million"]},
)

In [4]:
# Categories: states grouped by size
sized = states.assign(
    size=pd.cut(states["area_km2"], [0, 50_000, 150_000, 1e7], labels=["small", "medium", "large"]).astype(str)
)
kh.map(sized, "preset-choropleth fill-[size] ramp-set2", legend={"title": "State size"})

In [5]:
# Dark basemap and 3D, with the viridis ramp that holds up under lighting
kh.map(states, "fill-[density] ramp-viridis extrude-[density] basemap-dark highlight-amber-400", legend={"title": "People per km2"})

In [6]:
# Two layers on one map: muted states plus population points
points = states.assign(geometry=states.representative_point())
m = kh.Map()
m.add(states, "fill-slate-200/60 stroke-1 stroke-white legend-none")
m.add(points, "fill-[pop] ramp-magma-r size-7 stroke-white tooltip-[key,pop]", legend={"title": "Population", "fmt": "{:,.0f}"})
m

In [7]:
# Export: one HTML file, map fills the window, legend in its corner
out = Path("../spike_out")
out.mkdir(exist_ok=True)
final = kh.map(states, "preset-choropleth fill-[density] ramp-ylorrd tooltip-[key,density]", legend={"title": "People per km2", "note": SOURCE})
final.to_html(out / "sudan_density.html", title="Sudan population density")
webbrowser.open((out / "sudan_density.html").resolve().as_uri())

True

In [8]:
# Escape hatch: native Lonboard layers and the legend HTML for your own map
import ipywidgets as widgets
from lonboard import Map

layers = kh.layer(states, "fill-[density] ramp-ylorrd stroke-1 stroke-white")
widgets.VBox([Map(layers), widgets.HTML(kh.legend(states, "fill-[density] ramp-ylorrd", title="People per km2"))])